In [3]:
# Notebook cell: batch conversions between (ε,δ)-DP, μ-GDP, and (α, ε_α)-RDP

import math
from typing import Iterable, Optional

# =========================
# Normal CDF
# =========================
def normal_cdf(x: float) -> float:
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

# =========================
# GDP formulas
# =========================
def delta_from_eps_mu(eps: float, mu: float) -> float:
    """
    δ(ε; μ) = Φ(-ε/μ + μ/2) - exp(ε) Φ(-ε/μ - μ/2)
    """
    if mu <= 0:
        raise ValueError("mu must be > 0")
    term1 = normal_cdf(-eps / mu + mu / 2.0)
    term2 = math.exp(eps) * normal_cdf(-eps / mu - mu / 2.0)
    delta = term1 - term2
    return max(0.0, min(1.0, delta))

def mu_from_eps_delta(eps: float, delta_target: float, tol: float = 1e-12, max_iter: int = 300) -> float:
    """
    Solve for μ given ε and δ using bisection.
    """
    if eps < 0:
        raise ValueError("eps must be >= 0")
    if not (0 < delta_target < 1):
        raise ValueError("delta_target must be in (0,1)")

    lo = 1e-12
    hi = 1.0

    while delta_from_eps_mu(eps, hi) < delta_target:
        hi *= 2.0
        if hi > 1e6:
            raise RuntimeError(f"Could not bracket mu for eps={eps}, delta={delta_target}")

    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        val = delta_from_eps_mu(eps, mid)
        if abs(val - delta_target) < tol:
            return mid
        if val < delta_target:
            lo = mid
        else:
            hi = mid

    return 0.5 * (lo + hi)

def eps_from_mu_delta(mu: float, delta_target: float, tol: float = 1e-12, max_iter: int = 300) -> float:
    """
    Solve for ε given μ and δ using bisection.
    """
    if mu <= 0:
        raise ValueError("mu must be > 0")
    if not (0 < delta_target < 1):
        raise ValueError("delta_target must be in (0,1)")

    lo = 0.0
    hi = 1.0

    while delta_from_eps_mu(hi, mu) > delta_target:
        hi *= 2.0
        if hi > 1e6:
            raise RuntimeError(f"Could not bracket epsilon for mu={mu}, delta={delta_target}")

    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        val = delta_from_eps_mu(mid, mu)
        if abs(val - delta_target) < tol:
            return mid
        if val > delta_target:
            lo = mid
        else:
            hi = mid

    return 0.5 * (lo + hi)

def rdp_eps_from_mu_alpha(mu: float, alpha: float) -> float:
    """
    μ-GDP => (α, ε_α)-RDP with ε_α = α μ^2 / 2
    """
    if mu <= 0:
        raise ValueError("mu must be > 0")
    if alpha <= 1:
        raise ValueError("alpha must be > 1")
    return 0.5 * alpha * mu * mu

# =========================
# Pretty printer
# =========================
def print_table(rows, headers, float_digits=10):
    widths = []
    for i, h in enumerate(headers):
        col_vals = [h]
        for row in rows:
            v = row[i]
            if isinstance(v, float):
                col_vals.append(f"{v:.{float_digits}g}")
            else:
                col_vals.append(str(v))
        widths.append(max(len(x) for x in col_vals))

    def fmt(v, w):
        if isinstance(v, float):
            s = f"{v:.{float_digits}g}"
        else:
            s = str(v)
        return s.ljust(w)

    print(" | ".join(fmt(h, w) for h, w in zip(headers, widths)))
    print("-+-".join("-" * w for w in widths))
    for row in rows:
        print(" | ".join(fmt(v, w) for v, w in zip(row, widths)))

# =========================
# Batch converters
# =========================
def convert_eps_list_to_mu_rdp(
    eps_list: Iterable[float],
    delta: float,
    alpha: float = 2.0
):
    rows = []
    for eps in eps_list:
        mu = mu_from_eps_delta(eps, delta)
        rdp = rdp_eps_from_mu_alpha(mu, alpha)
        rows.append((eps, delta, mu, alpha, rdp))
    headers = ["eps", "delta", "mu", "alpha", "RDP_eps_alpha"]
    print("\nFrom (eps, delta) to mu-GDP and RDP:\n")
    print_table(rows, headers)
    return rows

def convert_mu_list_to_eps_rdp(
    mu_list: Iterable[float],
    delta: float,
    alpha: float = 2.0
):
    rows = []
    for mu in mu_list:
        eps = eps_from_mu_delta(mu, delta)
        rdp = rdp_eps_from_mu_alpha(mu, alpha)
        rows.append((mu, delta, eps, alpha, rdp))
    headers = ["mu", "delta", "eps_at_delta", "alpha", "RDP_eps_alpha"]
    print("\nFrom mu-GDP to (eps, delta)-DP and RDP:\n")
    print_table(rows, headers)
    return rows

# =========================
# EDIT THESE INPUTS
# =========================

# Case 1: insert eps values here
eps_values = [2.0, 4.38, 6.57, 10.0, 17.85]
delta = 1e-5
alpha = 5.0

# Case 2: insert mu values here
mu_values = [0.5, 1.0, math.sqrt(2), 2.0, math.sqrt(10)]

# =========================
# RUN BOTH
# =========================
eps_rows = convert_eps_list_to_mu_rdp(eps_values, delta=delta, alpha=alpha)
mu_rows  = convert_mu_list_to_eps_rdp(mu_values,  delta=delta, alpha=alpha)


From (eps, delta) to mu-GDP and RDP:

eps   | delta | mu           | alpha | RDP_eps_alpha
------+-------+--------------+-------+--------------
2     | 1e-05 | 0.5015516877 | 5     | 0.6288852386 
4.38  | 1e-05 | 1.000556797  | 5     | 2.502784759  
6.57  | 1e-05 | 1.413676493  | 5     | 4.996203066  
10    | 1e-05 | 2.000445619  | 5     | 10.00445669  
17.85 | 1e-05 | 3.161382064  | 5     | 24.98584139  

From mu-GDP to (eps, delta)-DP and RDP:

mu          | delta | eps_at_delta | alpha | RDP_eps_alpha
------------+-------+--------------+-------+--------------
0.5         | 1e-05 | 1.993091404  | 5     | 0.625        
1           | 1e-05 | 4.377178073  | 5     | 2.5          
1.414213562 | 1e-05 | 6.572970092  | 5     | 5            
2           | 1e-05 | 9.99725616   | 5     | 10           
3.16227766  | 1e-05 | 17.85656089  | 5     | 25           


In [4]:

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
device = 'cuda' if torch.cuda.is_available() else 'cpu'
num_gpus = 1 if device=='cuda' else 0
print(device)

cpu


In [ ]:
import numpy as np
from pathlib import Path

# Choose the parent folder and run name.
# Expected layout for this experiment:
# exp_data/max_grad_norm/1.0/seed5/cifar10_half_cnn_eps2.0/losses_in.npy
# exp_data/max_grad_norm/1.0/seed6/cifar10_half_cnn_eps2.0/losses_out.npy
results_root = Path("exp_data/max_grad_norm/1.0")
run_name = "cifar10_half_cnn_eps2.0"
seeds = [5, 6, 7, 8, 9]

all_losses_in = []
all_losses_out = []
emp_losses = []

for seed in seeds:
    base = results_root / f"seed{seed}" / run_name
    in_path = base / "losses_in.npy"
    out_path = base / "losses_out.npy"

    if not in_path.exists() or not out_path.exists():
        raise FileNotFoundError(f"Missing losses for seed {seed}: {base}")

    seed_losses_in = np.load(in_path).reshape(-1)
    seed_losses_out = np.load(out_path).reshape(-1)

    print(f"seed {seed}: losses_in {seed_losses_in.shape}, losses_out {seed_losses_out.shape}")

    all_losses_in.append(seed_losses_in)
    all_losses_out.append(seed_losses_out)
    emp_losses.append(np.mean(np.concatenate([seed_losses_in, seed_losses_out])))

losses_in = np.concatenate(all_losses_in)
losses_out = np.concatenate(all_losses_out)

print("\nConcatenated across seeds 5-9")
print("losses_in:", losses_in.shape, losses_in.dtype)
print("losses_out:", losses_out.shape, losses_out.dtype)
print("mean empirical loss across the five seeds:", float(np.mean(emp_losses)))
print(losses_in)
print(losses_out)

In [ ]:
class LoadDataset(torch.utils.data.Dataset):
    def __init__(self, N, x, y, x_dim, z_dim):
        self.x = x
        self.y = y
        # file imports and object initialization
        
        self.N = N
        self.x_dim = x_dim
        self.z_dim = z_dim

    def __getitem__(self, ix):
        a, b = self.x[ix,0:self.x_dim], self.y[ix,0:self.z_dim]
        
        return a, b
    
    def __len__(self):
        return self.N

In [ ]:
%reload_ext autoreload
from renyi import RenyiDivergenceEstimator
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import TQDMProgressBar

x_dim = 1 #plaintext length of a single sample
z_dim = 1 #ciphertext length of a single sample
N = 49999 # num samples -1
lr = 1e-4 #learning rate
epochs = 10 #epochs
batch_size = 5000 #batch size
Renyi_order = 2
accelerator = "gpu" if torch.cuda.is_available() else "cpu"
devices = 1  # one device either way

from pytorch_lightning.loggers import CSVLogger

csv_logger = CSVLogger("lightning_logs", name="renyi_run")

#format data
loss_type = ['mine']
# X, Y already constructed as tensors of shape (N, 1)
X = torch.tensor(losses_in,  dtype=torch.float32).view(-1, 1)
Y = torch.tensor(losses_out, dtype=torch.float32).view(-1, 1)
N = min(len(X), len(Y))
X, Y = X[:N], Y[:N]  # ensure same length

# --- 80/20 split: first 80% train, last 20% test ---
n_train = int(0.8 * N)
X_train, Y_train = X[:n_train], Y[:n_train]
X_test,  Y_test  = X[n_train:], Y[n_train:]

train_loader = torch.utils.data.DataLoader(
    LoadDataset(len(X_train), X_train, Y_train, x_dim=1, z_dim=1),
    batch_size=batch_size,
    shuffle=True,      # ok to shuffle *within* the training set
    drop_last=False
)

test_loader = torch.utils.data.DataLoader(
    LoadDataset(len(X_test), X_test, Y_test, x_dim=1, z_dim=1),
    batch_size=batch_size,
    shuffle=False,
    drop_last=False
)



trainer = Trainer(
    max_epochs=epochs,
    accelerator=accelerator,
    devices=devices,
    logger=csv_logger,
    enable_checkpointing=False,
    enable_progress_bar=True,
    callbacks=[TQDMProgressBar(refresh_rate=10)],
    log_every_n_steps=10,
)
model = RenyiDivergenceEstimator(
    lr=lr,
    renyi_order=Renyi_order,
    ema_rate=0.25,   # your EMA rate (beta)
    hidden=100,
)


trainer.fit(model, train_dataloaders=train_loader)
results = trainer.test(model, dataloaders=test_loader)

print("DV-Rényi lower bound (epoch-avg):", results[0]["test_renyi_lb"])
print("Same value from model:", model.avg_test_renyi_lb)